# Projection lithography, vectorially

`vw.load(name)` ships two real projection objectives, each carrying its
ray-traced exit-pupil wavefront so the package needs no ray tracer at run time:

| name | system | $\lambda$ | NA | medium |
|---|---|---|---|---|
| `euv` | US 7,151,592 six-mirror EUV projector | 13.4 nm | 0.22 | vacuum |
| `duv` | US 7,557,996 hyper-NA immersion objective | 193.4 nm | 1.2 | water ($n=1.60$) |

They resolve comparable features by opposite strategies: EUV uses a short
wavelength at modest aperture, so its focus is essentially **scalar**; DUV uses an
enormous aperture in immersion, so its focus is strongly **vectorial**.  This
notebook contrasts the two and images an actual layout through them.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")            # keep the rendered output tidy

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

import vectorwave as vw

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "figure.facecolor": "white"})
print("vectorwave", vw.__version__, "| packaged systems:", vw.available())

## 1 · The two systems

Both are diffraction-limited by the Maréchal criterion (RMS wavefront error below
$\lambda/14$), which is what makes their shipped wavefronts usable as-is.

In [ ]:
euv, duv = vw.load("euv"), vw.load("duv")

hdr = ["system", "lambda", "NA", "n", "WFE rms", "lambda/14", "diff.ltd",
       "Rayleigh hp", "Airy FWHM", "DoF"]
rows = []
for s in (euv, duv):
    rows.append([s.name.split()[0], f"{s.wavelength_nm:.1f} nm", f"{s.na:g}",
                 f"{s.n_image:.3f}", f"{s.wavefront_rms_nm:.3f} nm",
                 f"{s.wavelength_nm/14:.3f} nm", "yes" if s.diffraction_limited else "NO",
                 f"{s.rayleigh_half_pitch_nm:.1f} nm", f"{s.airy_fwhm_nm:.1f} nm",
                 f"{s.depth_of_focus_nm:.0f} nm"])
w = [max(len(h), max(len(r[i]) for r in rows)) for i, h in enumerate(hdr)]
print(" | ".join(h.ljust(w[i]) for i, h in enumerate(hdr)))
print("-+-".join("-" * x for x in w))
for r in rows:
    print(" | ".join(c.ljust(w[i]) for i, c in enumerate(r)))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
for a, s in zip(ax, (euv, duv)):
    W = s.wavefront.values * 1000.0                       # milliwaves
    m = np.abs(W).max()
    im = a.imshow(np.where(s.wavefront._inside, W, np.nan), extent=[-1, 1, -1, 1],
                  origin="lower", cmap="RdBu_r", vmin=-m, vmax=m)
    a.set_title(f"{s.name.split()[0]} exit-pupil wavefront\n"
                f"{s.wavefront.rms_waves*1000:.1f} m$\\lambda$ rms, "
                f"{s.wavefront.pv_waves*1000:.0f} m$\\lambda$ PV")
    a.set_xlabel("u"); a.set_ylabel("v")
    plt.colorbar(im, ax=a, shrink=0.85, label="m$\\lambda$")
fig.tight_layout()

## 2 · Point-spread functions

Each system's `pupil()` carries its traced wavefront.  The EUV focus is scalar for
all practical purposes; the DUV focus puts roughly a seventh of its energy in the
longitudinal component — which carries no image information, only background.
That is the whole reason hyper-NA lithography controls the illumination
polarization.

In [ ]:
def psf(system, half=2.0, step=1 / 44, polarization="x"):
    pupil = system.pupil(polarization=polarization)
    grid = vw.Grid.from_spacing(0.25, 256)
    xr = np.arange(-half, half + 1e-9, step)
    return pupil, pupil.spectrum(grid).field_on(xr, xr, 0.0), xr

pupil_euv, f_euv, x_euv = psf(euv)
pupil_duv, f_duv, x_duv = psf(duv)

for tag, s, p, f in (("EUV", euv, pupil_euv, f_euv), ("DUV", duv, pupil_duv, f_duv)):
    fr = f.component_fractions()
    print(f"{tag}: FWHM {f.fwhm('x'):.3f} lambda -> {f.fwhm('x')*s.wavelength_nm:5.1f} nm "
          f"[Airy {s.airy_fwhm_nm:.1f} nm]   |Ez|^2 = {fr['z']*100:5.2f} %")

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(13, 6.2))
for row, (tag, s, f, xr) in enumerate((("EUV", euv, f_euv, x_euv),
                                       ("DUV", duv, f_duv, x_duv))):
    ext = [xr[0], xr[-1], xr[0], xr[-1]]
    comps = [("total $|E|^2$", f.intensity), ("$|E_x|^2$", np.abs(f.Ex) ** 2),
             ("$|E_y|^2$  (cross)", np.abs(f.Ey) ** 2),
             ("$|E_z|^2$  (long.)", np.abs(f.Ez) ** 2)]
    for col, (title, I) in enumerate(comps):
        a = ax[row, col]
        a.imshow(I, extent=ext, origin="lower", cmap="inferno", norm=PowerNorm(0.5))
        share = I.sum() / f.intensity.sum()
        a.set_title(f"{tag} · {title}" + ("" if col == 0 else f"  ({share*100:.2f} %)"))
        a.set_xlim(-1.5, 1.5); a.set_ylim(-1.5, 1.5); a.set_xlabel("x / $\\lambda$")
        if col == 0:
            a.set_ylabel("y / $\\lambda$")
fig.tight_layout()

## 3 · Imaging a real layout

Now an extended object with no symmetry — a Manhattan routing layout shipped with
the repo — imaged at wafer scale through the DUV objective with partially coherent
illumination (a conventional disc source, $\sigma = 0.6$).  The image is kept
split by field component, so we can read off the longitudinal background directly.

In [ ]:
im = duv.imaging(pixel_nm=12.0, size=512, polarization="x")
mask = vw.Mask.from_image("assets/circuit_pattern.png", pixel=12.0, size=512)
aerial = im.aerial_image(mask, sigma=0.6, source_points=9, vector=True)
fr = aerial.fractions()
print(f"aerial-image energy split:  Ex {fr['x']:.3f}   Ey {fr['y']:.4f}   Ez {fr['z']:.3f}")
print(f"image contrast (Michelson): {aerial.contrast():.3f}")

extent = mask.grid.extent
extent_um = [e / 1000 for e in extent]                    # nm -> um for the axes
fig, ax = plt.subplots(1, 3, figsize=(13, 4.2))
ax[0].imshow(mask.data, extent=extent_um, origin="lower", cmap="gray")
ax[0].set_title("mask (wafer scale)")
ax[1].imshow(aerial.normalized, extent=extent_um, origin="lower", cmap="inferno")
ax[1].set_title(f"vector aerial image ($\\sigma$=0.6)\ncontrast {aerial.contrast():.2f}")
ax[2].imshow(aerial.Iz / aerial.total.max(), extent=extent_um, origin="lower", cmap="magma")
ax[2].set_title(f"longitudinal $I_z$  ({fr['z']*100:.1f} % of energy)")
for a in ax:
    a.set_xlabel("x / $\\mu$m"); a.set_ylabel("y / $\\mu$m")
fig.tight_layout()

The longitudinal channel is not noise — it is a structured background that tracks
the mask edges, strongest where the local features run along the polarization
direction.  A scalar model cannot see it.

## 4 · Resolution and the polarization penalty

Image equal line/space gratings of shrinking half pitch and read the contrast.
For dense lines at hyper-NA the illumination polarization matters: **TE**
(polarized along the lines) keeps the diffracted orders' fields parallel and
prints with high contrast, while **TM** (across the lines) lets the orders'
fields tip apart by $\cos 2\theta$ and washes out.  The scalar model is blind to
the difference.

In [ ]:
hp = np.linspace(120, 34, 12)                             # half pitch, nm

def contrast_curve(polarization, vector):
    sysm = duv.imaging(pixel_nm=8.0, size=256, polarization=polarization)
    return sysm.contrast_vs_pitch(hp, sigma=0.5, source_points=7, vector=vector)

c_te = contrast_curve("y", True)                          # lines vertical -> E along lines is 'y'
c_tm = contrast_curve("x", True)
c_sc = contrast_curve("x", False)

fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.plot(hp, c_te, "o-", lw=1.8, ms=3, label="TE (E along lines)")
ax.plot(hp, c_tm, "s-", lw=1.8, ms=3, label="TM (E across lines)")
ax.plot(hp, c_sc, "k--", lw=1.3, label="scalar (blind to polarization)")
ax.axvline(duv.rayleigh_half_pitch_nm, color="grey", ls=":", lw=1,
           label=f"Rayleigh {duv.rayleigh_half_pitch_nm:.0f} nm")
ax.set_xlabel("half pitch (nm)"); ax.set_ylabel("contrast")
ax.set_title("DUV NA 1.2: line/space contrast vs pitch"); ax.invert_xaxis()
ax.legend(fontsize=8); ax.grid(alpha=0.25)
fig.tight_layout()

i = int(np.argmin(np.abs(hp - 60)))
print(f"at {hp[i]:.0f} nm half pitch:  TE {c_te[i]:.3f}   TM {c_tm[i]:.3f}   scalar {c_sc[i]:.3f}")

## 5 · Scalar and vector agree where they should

The vector machinery is not a different physics — it is a *superset*.  Drop the NA
low enough and the vector and scalar aerial images become indistinguishable; the
longitudinal component vanishes and the projection factors go to unity.

In [ ]:
low = vw.ImagingSystem(na=0.10, wavelength=193.0, pixel=200.0, size=128)
m = vw.Mask.lines_spaces(2000.0, pixel=200.0, size=128)
v = low.aerial_image(m, vector=True).normalized
s = low.aerial_image(m, vector=False).normalized
print(f"max |vector - scalar| at NA 0.1: {np.abs(v - s).max():.2e}  (indistinguishable)")

line = v.shape[0] // 2
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(m.grid.x / 1000, v[line], lw=2.2, label="vector")
ax.plot(m.grid.x / 1000, s[line], "k--", lw=1.2, label="scalar")
ax.set_xlabel("x / $\\mu$m"); ax.set_ylabel("normalized intensity")
ax.set_title("NA 0.10: the vector model reduces to the scalar one")
ax.legend(fontsize=8); ax.grid(alpha=0.25)
fig.tight_layout()

That is the invariant worth keeping in view: the same code that reduces to the
textbook Airy pattern and the scalar aerial image at low NA is the one that, at
NA 1.2 in immersion, tells you how much of your image is longitudinal background
and how much contrast your polarization choice costs.